In [5]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.insert(0, "/home/s2864332/MySYNGAP/ArtifactDetection")
import artifactdetection as ad
print(ad.__file__)

/home/s2864332/MySYNGAP/ArtifactDetection/artifactdetection/__init__.py


### Make sure recording files are in .npy and brain state files are in .pkl before applying artifactdetection functions

Apply the following functions to convert recording files from .dat files to .npy, and brainstate files from .xls to .pkl

### 1. Convert .dat to .npy using the Preprocess class

In [ ]:
#folder with unformatted .dat files 
raw_rec_folder = "/home/s2864332/SYNGAP_Rat_Data/raw/raw_recordings_all" #folder path to unformatted files
raw_file = [file for file in os.listdir(raw_rec_folder)] #list of file names to feed to preprocess
print(len(raw_file))
save_path = '/home/s2864332/SYNGAP_Rat_Data/npy_format/formatted_rec' #folder path to save files
if not os.path.exists(save_path):
    os.makedirs(save_path)

18


#### Loop through the files in the raw_file list and select an identifier to save the file as (e.g animal id)

In [ ]:
for file in raw_file:
    animal_id = file[0:5] #keep track of animal id of original .dat file and pass to save_as argument to save the correct animal id
    if animal_id == '.ipyn': 
        pass
    else:
        print(animal_id)
        preprocess = ad.Preprocess(raw_folder = raw_rec_folder, raw_file = file, save_fold_path = save_path, save_as = animal_id)
        preprocess.convert_dat_to_npy()

S7075
data saved for S7075_Basline1-2020_03_24-0000.dat
S7074
data saved for S7074_Basline1-2020_03_24-0000.dat
S7069
data saved for S7069_Baseline-2020_02_10-0000.dat
S7064
data saved for S7064_Baseline1-2020_01_27-0000.dat
S7071
data saved for S7071_Baseline-2020_03_16-0000.dat
S7070
data saved for S7070_Baseline-2020_03_16-0000.dat
S7063
data saved for S7063_Baseline1-2020_01_27-0000.dat
S7068
data saved for S7068_Baseline-2020_02_10-0000.dat
S7072
data saved for S7072_Baseline-2020_03_16-0000.dat


### 2. Convert .xls to .pkl 

In [6]:
#folder with unformatted brain state (.xls) files 
unformatted_br_folder = '/home/s2864332/SYNGAP_Rat_Data/raw/raw_brain_states'
unformatted_file = [file for file in os.listdir(unformatted_br_folder)] #list of file names to feed to preprocess
save_br_path = '/home/s2864332/SYNGAP_Rat_Data/npy_format/formatted_br' #folder path to save files
if not os.path.exists(save_br_path):
    os.makedirs(save_br_path)

In [2]:
for file in unformatted_file:
    print(file)
    br_id = file[0:9]
    if file.endswith(".xls"):
        ad.reformat_br_file(unformatted_folder = unformatted_br_folder, unformatted_file = file, save_folder = save_br_path,
                       save_as = br_id)
    


NameError: name 'unformatted_file' is not defined

In [3]:
manual_path = '/home/melissa/PREPROCESSING/SYNGAP1/numpyformat_baseline'
code_path = save_br_path

for xls_file_name in os.listdir(unformatted_br_folder):   
    print(50*'=') 
    file_name = xls_file_name[0:9] + '.pkl'
    print(file_name)

    xsl_file_path = os.path.join(unformatted_br_folder, xls_file_name)
    manual_file_path = os.path.join(manual_path, file_name)
    code_file_path =  os.path.join(code_path, file_name)

    if not os.path.exists(manual_file_path):
        print(f'Manual file for {file_name} does not exist. Skipping comparison.')
        continue

    if not os.path.exists(code_file_path):
        print(f'Code file for {file_name} does not exist. Skipping comparison.')
        continue
    
    xsl_file = pd.read_excel(xsl_file_path)
    manual_pkl_file = pd.read_pickle(manual_file_path)
    code_pkl_file = pd.read_pickle(code_file_path)

    print('Manually fixed file:')
    print(manual_pkl_file.head())
    print(20*'-')
    print('Code fixed file:')
    print(code_pkl_file.head()) 
    print(20*'-')
    print('Main xls file:')
    print(xsl_file.head())
    print(len(xsl_file), len(manual_pkl_file), len(code_pkl_file))


NameError: name 'save_br_path' is not defined

### Sanity Check:
Checking produced pkl files using the *reformat_br_file* function and comparing to the hand data. Specially in terms of **brain state** for the first epoch. 

In [4]:
import os
import pandas as pd


path_a = "/home/s2864332/MySYNGAP/Preprocess/ArtifactDetectTest/preparing_data/formatted_br"     
path_b = "/home/s2864332/MySYNGAP/Preprocess/ArtifactDetectTest/manual_brain_states_from_Lucy"   
path_b = '/home/melissa/PREPROCESSING/SYNGAP1/numpyformat_baseline'  
output_csv = "/home/s2864332/MySYNGAP/Preprocess/ArtifactDetectTest/preparing_data/melissa_comparison_report.csv"

results = []
print(len(os.listdir(path_b)), "files found in B")
print(len(os.listdir(path_a)), "files found in A")

for fname in sorted(os.listdir(path_a)):
    if not fname.lower().endswith(".pkl"):
        continue

    file_a = os.path.join(path_a, fname)
    file_b = os.path.join(path_b, fname)

    if not os.path.exists(file_b):
        results.append({
            "file": fname,
            "status": "missing_in_B",
        })
        continue

    if not os.path.exists(file_a):
        results.append({
            "file": fname,
            "status": "missing_in_A",
        })
        continue

    try:
        df_a = pd.read_pickle(file_a)
        df_b = pd.read_pickle(file_b)
    except Exception as e:
        results.append({
            "file": fname,
            "status": f"error_loading ({e})"
        })
        continue

    try:
        col_a = df_a["brainstate"].astype(int).reset_index(drop=True)
        col_b = df_b["brainstate"].astype(int).reset_index(drop=True)
    except Exception:
        results.append({
            "file": fname,
            "status": "missing_brainstate_column",
            "match": False,
            "len_A": None,
            "len_B": None
        })
        continue

    # Compare entire column
    columns_match = col_a.equals(col_b)

    results.append({
        "file": fname,
        "status": "match" if columns_match else "different",
        "match": columns_match,
        "len_A": len(col_a),
        "len_B": len(col_b)
    })

results_df = pd.DataFrame(results)
print(results_df)

results_df.to_csv(output_csv, index=False)
print(f"\nComparison report saved to: {output_csv}")


57 files found in B


FileNotFoundError: [Errno 2] No such file or directory: '/home/s2864332/MySYNGAP/Preprocess/ArtifactDetectTest/preparing_data/formatted_br'